In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter, freqz
from ipywidgets import FloatSlider, IntSlider, HBox, HTML, Layout, interactive_output
from IPython.display import display

# ============================================================
# FROM SIGNAL-FLOW GRAPH TO TRANSFER FUNCTION
# ============================================================

plt.close('all')

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.sfg-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.sfg-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.sfg-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:9px 13px;
    border-radius:0 0 8px 8px;
    font-size:14px;
    line-height:1.45;
    margin-bottom:7px;
}

.sfg-title{
    color:#0d47a1;
    font-weight:bold;
    font-size:14.5px;
    margin-bottom:5px;
}

.sfg-equation{
    text-align:center;
    font-family:serif;
    font-size:16px;
    margin:6px 0;
}

.sfg-stage{
    background:#fff8e6;
    border:1px solid #d8b451;
    border-radius:7px;
    padding:9px 12px;
    font-size:14px;
    line-height:1.50;
}

.sfg-system{
    width:100%;
    box-sizing:border-box;
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:7px 12px;
    margin:0 0 7px 0;
    font-size:14px;
    line-height:1.45;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="sfg-root">

<div class="sfg-header">
From Signal-Flow Graph to Transfer Function
</div>

<div class="sfg-doc">

A signal-flow graph is an alternative representation of a difference equation.
The graph studied in this notebook implements the first-order recursive system
whose transfer function is

<div class="sfg-equation">
<b>
H(z) = Y(z)/X(z) =
(β₀ + β₁z⁻¹)/(1 − αz⁻¹).
</b>
</div>

Thus, <b>α</b> determines the recursive feedback term, whereas
<b>β₀</b> and <b>β₁</b> determine the feedforward part of the system.

The purpose of the notebook is to show how this transfer function follows
directly from the corresponding signal-flow graph:

<div style="text-align:center;font-size:14.5px;margin:7px 0;">
<b>
Signal-flow graph → Node equations → Combined equations → Z-transform → H(z)
</b>
</div>

The <b>Stage</b> control acts on both the mathematical derivation and the graph.
At every stage, the branches involved in the current step are emphasized,
while the remaining branches remain visible in a faded form.

The sliders α, β₀ and β₁ change the branch weights and therefore modify the
system itself. The corresponding impulse response and magnitude response are
calculated automatically.

For this demonstration, |α| &lt; 1 so that the recursive system remains stable.

</div>

</div>
"""))

# ============================================================
# CONTROLS
# ============================================================

stage_slider = IntSlider(value=1,min=1,max=5,step=1,description='Stage:',continuous_update=True,style={'description_width':'45px'},layout=Layout(width='210px'))

alpha_slider = FloatSlider(value=0.60,min=-0.95,max=0.95,step=0.01,description='α:',continuous_update=True,readout_format='.2f',style={'description_width':'20px'},layout=Layout(width='210px'))

beta0_slider = FloatSlider(value=1.00,min=-2.00,max=2.00,step=0.05,description='β₀:',continuous_update=True,readout_format='.2f',style={'description_width':'25px'},layout=Layout(width='210px'))

beta1_slider = FloatSlider(value=0.50,min=-2.00,max=2.00,step=0.05,description='β₁:',continuous_update=True,readout_format='.2f',style={'description_width':'25px'},layout=Layout(width='210px'))

controls = HBox([stage_slider,alpha_slider,beta0_slider,beta1_slider],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b9cce5',padding='7px 8px',margin='0 0 7px 0'))

# ============================================================
# CURRENT SYSTEM
# ============================================================

def current_system(alpha,beta0,beta1):

    return f"""
    <div class="sfg-root">

    <div class="sfg-system">

    <div class="sfg-title">Current transfer function</div>

    <div class="sfg-equation">
    H(z) =
    ({beta0:.2f} {beta1:+.2f}z⁻¹) /
    (1 {(-alpha):+.2f}z⁻¹)
    </div>

    </div>

    </div>
    """

system_output = interactive_output(current_system,{'alpha':alpha_slider,'beta0':beta0_slider,'beta1':beta1_slider})

# ============================================================
# STAGE DOCUMENTATION
# ============================================================

def stage_text(stage,alpha,beta0,beta1):

    if stage == 1:

        return f"""
        <div class="sfg-root">
        <div class="sfg-stage">

        <div class="sfg-title">Stage 1 — Read the node equations</div>

        The complete signal-flow graph is active. Reading the incoming branches
        at each node gives

        <div class="sfg-equation">
        w₁[n] = αw₄[n] + x[n]
        </div>

        <div class="sfg-equation">
        w₂[n] = w₁[n]
        </div>

        <div class="sfg-equation">
        w₃[n] = β₀w₂[n] + β₁w₄[n]
        </div>

        <div class="sfg-equation">
        w₄[n] = w₂[n−1]
        </div>

        <div class="sfg-equation">
        y[n] = w₃[n]
        </div>

        Current values:
        <b>α = {alpha:.2f}</b>,
        <b>β₀ = {beta0:.2f}</b>,
        <b>β₁ = {beta1:.2f}</b>.

        </div>
        </div>
        """

    elif stage == 2:

        return """
        <div class="sfg-root">
        <div class="sfg-stage">

        <div class="sfg-title">Stage 2 — Combine the node equations</div>

        The graph emphasizes the branches that allow the intermediate variables
        to be eliminated.

        <div class="sfg-equation">
        w₂[n] = w₁[n], &nbsp;&nbsp;
        w₄[n] = w₂[n−1]
        </div>

        leading to

        <div class="sfg-equation">
        <b>w₂[n] = αw₂[n−1] + x[n]</b>
        </div>

        and

        <div class="sfg-equation">
        <b>y[n] = β₀w₂[n] + β₁w₂[n−1].</b>
        </div>

        </div>
        </div>
        """

    elif stage == 3:

        return """
        <div class="sfg-root">
        <div class="sfg-stage">

        <div class="sfg-title">Stage 3 — Isolate the recursive part</div>

        Only the branches associated with the feedback recursion are emphasized.

        <div class="sfg-equation">
        w₂[n] = αw₂[n−1] + x[n]
        </div>

        Taking the Z-transform,

        <div class="sfg-equation">
        W₂(z) = αz⁻¹W₂(z) + X(z)
        </div>

        and therefore

        <div class="sfg-equation">
        <b>
        W₂(z) = X(z)/(1 − αz⁻¹).
        </b>
        </div>

        </div>
        </div>
        """

    elif stage == 4:

        return """
        <div class="sfg-root">
        <div class="sfg-stage">

        <div class="sfg-title">Stage 4 — Isolate the output part</div>

        The branches carrying β₀ and β₁ toward the output are now emphasized.

        <div class="sfg-equation">
        y[n] = β₀w₂[n] + β₁w₂[n−1]
        </div>

        and therefore

        <div class="sfg-equation">
        Y(z) = (β₀ + β₁z⁻¹)W₂(z).
        </div>

        Substitution of W₂(z) gives

        <div class="sfg-equation">
        Y(z) =
        [(β₀ + β₁z⁻¹)/(1 − αz⁻¹)]X(z).
        </div>

        </div>
        </div>
        """

    else:

        return f"""
        <div class="sfg-root">
        <div class="sfg-stage">

        <div class="sfg-title">Stage 5 — Complete transfer function</div>

        The recursive and feedforward parts together give

        <div class="sfg-equation">
        <b>
        H(z) =
        Y(z)/X(z) =
        (β₀ + β₁z⁻¹)/(1 − αz⁻¹).
        </b>
        </div>

        With the current coefficient values,

        <div class="sfg-equation">
        <b>
        H(z) =
        ({beta0:.2f} {beta1:+.2f}z⁻¹) /
        (1 {(-alpha):+.2f}z⁻¹).
        </b>
        </div>

        The pole is located at

        <div class="sfg-equation">
        z = α = <b>{alpha:.2f}</b>.
        </div>

        </div>
        </div>
        """

stage_output = interactive_output(stage_text,{'stage':stage_slider,'alpha':alpha_slider,'beta0':beta0_slider,'beta1':beta1_slider})

# ============================================================
# SIGNAL-FLOW GRAPH
# ============================================================

def draw_signal_flow_graph(ax,stage,alpha,beta0,beta1):

    ax.set_xlim(-1.10,6.15)
    ax.set_ylim(-2.15,1.95)
    ax.axis('off')

    nodes = {
        'x':(-0.65,0.55),
        'w1':(1.00,0.55),
        'w2':(2.35,0.55),
        'w3':(3.90,0.55),
        'y':(5.55,0.55),
        'w4':(2.35,-1.15)
    }

    branch_names = ['x_w1','w1_w2','w2_w3','w3_y','w2_w4','w4_w3','w4_w1']

    if stage == 1:
        active = set(branch_names)

    elif stage == 2:
        active = {'x_w1','w1_w2','w2_w4','w4_w1','w2_w3','w4_w3','w3_y'}

    elif stage == 3:
        active = {'x_w1','w1_w2','w2_w4','w4_w1'}

    elif stage == 4:
        active = {'w2_w4','w2_w3','w4_w3','w3_y'}

    else:
        active = set(branch_names)

    def branch_style(name):

        if name in active:
            return {'arrowstyle':'->','linewidth':2.0,'color':'tab:red','alpha':1.0,'shrinkA':5,'shrinkB':5}

        return {'arrowstyle':'->','linewidth':1.0,'color':'0.75','alpha':0.35,'shrinkA':5,'shrinkB':5}

    # --------------------------------------------------------
    # NODES
    # --------------------------------------------------------

    for name,(x0,y0) in nodes.items():

        if name == 'x':

            ax.text(-0.92,0.55,r'$x[n]$',ha='center',va='center',fontsize=10.5,fontweight='bold')

        elif name == 'y':

            ax.text(5.88,0.55,r'$y[n]$',ha='center',va='center',fontsize=10.5,fontweight='bold')

        else:

            circle = plt.Circle((x0,y0),0.115,fill=False,linewidth=1.2,color='black')

            ax.add_patch(circle)

    # --------------------------------------------------------
    # NODE LABELS
    # --------------------------------------------------------

    ax.text(1.00,1.28,r'$w_1[n]$',ha='center',va='center',fontsize=9.5)

    ax.text(2.35,1.28,r'$w_2[n]$',ha='center',va='center',fontsize=9.5)

    ax.text(3.90,1.28,r'$w_3[n]$',ha='center',va='center',fontsize=9.5)

    ax.text(2.35,-1.65,r'$w_4[n]$',ha='center',va='center',fontsize=9.5)

    # --------------------------------------------------------
    # x -> w1
    # --------------------------------------------------------

    ax.annotate('',xy=nodes['w1'],xytext=(-0.55,0.55),arrowprops=branch_style('x_w1'))

    # --------------------------------------------------------
    # w1 -> w2
    # --------------------------------------------------------

    ax.annotate('',xy=nodes['w2'],xytext=nodes['w1'],arrowprops=branch_style('w1_w2'))

    # --------------------------------------------------------
    # w2 -> w3
    # --------------------------------------------------------

    ax.annotate('',xy=nodes['w3'],xytext=nodes['w2'],arrowprops=branch_style('w2_w3'))

    label_alpha = 1.0 if 'w2_w3' in active else 0.30

    ax.text(3.12,0.96,rf'$\beta_0={beta0:.2f}$',ha='center',va='center',fontsize=9,alpha=label_alpha)

    # --------------------------------------------------------
    # w3 -> y
    # --------------------------------------------------------

    ax.annotate('',xy=(5.46,0.55),xytext=nodes['w3'],arrowprops=branch_style('w3_y'))

    # --------------------------------------------------------
    # w2 -> w4
    # --------------------------------------------------------

    ax.annotate('',xy=nodes['w4'],xytext=nodes['w2'],arrowprops=branch_style('w2_w4'))

    label_alpha = 1.0 if 'w2_w4' in active else 0.30

    ax.text(2.60,0.02,r'$z^{-1}$',ha='left',va='center',fontsize=10,alpha=label_alpha)

    # --------------------------------------------------------
    # w4 -> w3
    # --------------------------------------------------------

    ax.annotate('',xy=nodes['w3'],xytext=nodes['w4'],arrowprops=branch_style('w4_w3'))

    label_alpha = 1.0 if 'w4_w3' in active else 0.30

    ax.text(3.65,-0.55,rf'$\beta_1={beta1:.2f}$',ha='left',va='center',fontsize=9,alpha=label_alpha)

    # --------------------------------------------------------
    # w4 -> w1 FEEDBACK
    # --------------------------------------------------------

    feedback_style = branch_style('w4_w1')

    feedback_style['connectionstyle'] = 'arc3,rad=-0.30'

    ax.annotate('',xy=nodes['w1'],xytext=nodes['w4'],arrowprops=feedback_style)

    label_alpha = 1.0 if 'w4_w1' in active else 0.30

    ax.text(0.42,-0.82,rf'$\alpha={alpha:.2f}$',ha='right',va='center',fontsize=9,alpha=label_alpha)

    # --------------------------------------------------------
    # STAGE LABEL
    # --------------------------------------------------------

    stage_labels = {
        1:'Complete graph — read all node equations',
        2:'Combine the intermediate node equations',
        3:'Recursive path used to obtain W₂(z)',
        4:'Output path used to obtain Y(z)',
        5:'Complete graph — final H(z)'
    }

    ax.text(2.45,-2.00,stage_labels[stage],ha='center',va='center',fontsize=9.2,fontweight='bold')

    ax.set_title('Signal-Flow Graph')

# ============================================================
# RESULTS
# ============================================================

def draw_results(stage,alpha,beta0,beta1):

    b = np.array([beta0,beta1])

    a = np.array([1.0,-alpha])

    n = np.arange(0,30)

    delta = np.zeros(len(n))

    delta[0] = 1.0

    h = lfilter(b,a,delta)

    omega,H = freqz(b,a,worN=1024)

    magnitude_db = 20.0*np.log10(np.maximum(np.abs(H),1e-12))

    fig,axes = plt.subplots(1,3,figsize=(9.0,3.55))

    ax1,ax2,ax3 = axes

    # --------------------------------------------------------
    # SIGNAL-FLOW GRAPH
    # --------------------------------------------------------

    draw_signal_flow_graph(ax1,stage,alpha,beta0,beta1)

    # --------------------------------------------------------
    # IMPULSE RESPONSE
    # --------------------------------------------------------

    markerline,stemlines,baseline = ax2.stem(n,h,linefmt='r-',markerfmt='ro',basefmt=' ')

    plt.setp(stemlines,linewidth=1.0)

    markerline.set_markersize(3.5)

    ax2.axhline(0,linewidth=0.8)

    ax2.set_xlim(-0.5,29.5)

    maximum_h = max(1.0,np.max(np.abs(h))*1.15)

    ax2.set_ylim(-maximum_h,maximum_h)

    ax2.set_title('Impulse Response')

    ax2.set_xlabel('Sample index n')

    ax2.set_ylabel('h[n]')

    ax2.grid(True,linestyle=':',alpha=0.25)

    # --------------------------------------------------------
    # MAGNITUDE RESPONSE
    # --------------------------------------------------------

    ax3.plot(omega/np.pi,magnitude_db,color='red',linewidth=1.4)

    ax3.set_xlim(0,1)

    lower_limit = min(-60.0,np.min(magnitude_db)-5.0)

    upper_limit = max(10.0,np.max(magnitude_db)+5.0)

    ax3.set_ylim(lower_limit,upper_limit)

    ax3.set_title('Magnitude Response')

    ax3.set_xlabel(r'Normalized frequency $\omega/\pi$')

    ax3.set_ylabel('Magnitude (dB)')

    ax3.grid(True,linestyle=':',alpha=0.25)

    plt.subplots_adjust(left=0.055,right=0.98,top=0.87,bottom=0.20,wspace=0.34)

    plt.show()

    plt.close(fig)

plot_output = interactive_output(draw_results,{'stage':stage_slider,'alpha':alpha_slider,'beta0':beta0_slider,'beta1':beta1_slider})

# ============================================================
# DISPLAY
# ============================================================

display(controls)

display(system_output)

display(stage_output)

display(plot_output)